# Create a Foundry IQ Knowledge Base

In this notebook, you will create a **Foundry IQ knowledge base** programmatically using Python. A knowledge base connects your data (documents, files, URLs) to Azure AI Search so your agents can retrieve relevant information when answering questions.

By the end, you will have:
- Created an Azure Storage Account and uploaded documents
- Created an Azure AI Search service
- Assigned the RBAC roles required for identity-based access
- Built an Azure AI Search index, data source, and indexer that connect to your documents
- Created the knowledge source and knowledge base and printed the MCP endpoint for agent queries
- Registered the knowledge base with your Foundry project so it appears in the portal's Knowledge (Foundry IQ) blade
- Verified the index is populated and returning relevant results

---

### Prerequisites

- An active Azure subscription
- Azure CLI installed and authenticated (`az login`)
- An Azure AI Foundry project (hub + project) — you can create one in the [Azure AI Foundry portal](https://ai.azure.com/)
- A deployed model in your Foundry project (e.g., `gpt-5.1`)
- Python 3.12

### Step 1: Install Packages

Run the cell below to install the Python libraries this notebook needs. You only need to do this once per environment.

In [ ]:
# Azure AI Projects SDK, identity, storage, search management and data-plane SDKs
# NOTE: Use Python 3.12 — Python 3.14 has typing incompatibilities with the openai package
%pip install azure-ai-projects==2.2.0 openai==2.9.0 python-dotenv azure-identity \
    azure-storage-blob azure-mgmt-storage azure-mgmt-search azure-search-documents

### Step 2: Load Configuration

All connection details are read from a `.env` file so that secrets never appear in source code.

> **Fill in `.env` first.** Open the `.env` file in this folder and replace every `<...>` placeholder with your own value before running this cell. Entries that already have a value have working defaults — leave them as they are.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file in the current directory.
# override=True so that editing .env and re-running this cell picks up the new
# values instead of keeping stale ones already loaded into the kernel.
load_dotenv(override=True)

# Read each value from the environment
endpoint            = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model               = os.getenv("MODEL_DEPLOYMENT_NAME")
subscription_id     = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group      = os.getenv("AZURE_RESOURCE_GROUP")
storage_account_name = os.getenv("STORAGE_ACCOUNT_NAME")
search_service_name = os.getenv("SEARCH_SERVICE_NAME")
container_name      = os.getenv("BLOB_CONTAINER_NAME", "knowledge-source")
index_name          = os.getenv("SEARCH_INDEX_NAME", "support-kb-index")
data_source_name    = os.getenv("SEARCH_DATA_SOURCE_NAME", "support-kb-blob-source")
indexer_name        = os.getenv("SEARCH_INDEXER_NAME", "support-kb-indexer")
knowledge_source_name = os.getenv("KNOWLEDGE_SOURCE_NAME", "support-kb-source")
knowledge_base_name = os.getenv("KNOWLEDGE_BASE_NAME", "support-kb")

# Fail fast with a clear message if a required value is missing or still a
# placeholder. Without this guard a missing value surfaces much later as a
# cryptic SDK error (e.g. "Parameter 'subscription_id' must not be None").
required = {
    "FOUNDRY_PROJECT_ENDPOINT": endpoint,
    "MODEL_DEPLOYMENT_NAME": model,
    "AZURE_SUBSCRIPTION_ID": subscription_id,
    "AZURE_RESOURCE_GROUP": resource_group,
    "STORAGE_ACCOUNT_NAME": storage_account_name,
    "SEARCH_SERVICE_NAME": search_service_name,
}
missing = [key for key, value in required.items() if not value or "<" in value]
if missing:
    raise ValueError(
        "Missing required configuration: " + ", ".join(missing) + ".\n"
        "Copy .env.example to .env and replace every <...> placeholder with your "
        "own values before running this cell."
    )

print(f"Foundry endpoint  : {endpoint}")
print(f"Model             : {model}")
print(f"Subscription      : {subscription_id}")
print(f"Resource group    : {resource_group}")
print(f"Storage account   : {storage_account_name}")
print(f"Search service    : {search_service_name}")
print(f"Knowledge base    : {knowledge_base_name}")

### Step 3: Connect to Azure

We use `DefaultAzureCredential` for all authentication throughout this notebook. This picks up your Azure CLI login automatically — no API keys needed.

In [ ]:
from azure.identity import DefaultAzureCredential

# Single credential object reused across all SDK clients
credential = DefaultAzureCredential()
print("Credential initialized — using DefaultAzureCredential")

### Step 4: Create a Storage Account

We use the Azure Management SDK to create a Storage Account (Standard LRS, StorageV2) and a blob container configured in `.env` where our documents will live.

> **Note:** Storage account names must be globally unique, lowercase, and between 3 and 24 characters. If creation fails with a naming conflict, change `STORAGE_ACCOUNT_NAME` in your `.env` file.

In [ ]:
from azure.mgmt.storage import StorageManagementClient
from azure.mgmt.storage.models import (
    StorageAccountCreateParameters,
    Sku,
    Kind,
)

# Read the Azure region from the environment (defaults to eastus2 if not set)
location = os.getenv("AZURE_LOCATION", "eastus2")

# Create a management client for Storage operations
storage_mgmt_client = StorageManagementClient(credential, subscription_id)

# Create the storage account (this is a long-running operation)
print(f"Creating storage account '{storage_account_name}'...")
poller = storage_mgmt_client.storage_accounts.begin_create(
    resource_group_name=resource_group,
    account_name=storage_account_name,
    parameters=StorageAccountCreateParameters(
        sku=Sku(name="Standard_LRS"),
        kind=Kind.STORAGE_V2,
        location=location,
    ),
)
storage_account = poller.result()
print(f"Storage account created: {storage_account.name}")
print(f"  Location: {storage_account.location}")
print(f"  Provisioning state: {storage_account.provisioning_state}")

In [ ]:
# Create a blob container inside the storage account.
# The signed-in user also needs a data-plane role before blob uploads work.
import subprocess
import shutil
import time
from azure.storage.blob import BlobServiceClient

AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

user_result = subprocess.run(
    [AZ, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True,
    text=True,
    encoding="utf-8",
)
user_id = user_result.stdout.strip()

storage_scope = (
    f"/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}"
    f"/providers/Microsoft.Storage/storageAccounts/{storage_account_name}"
)

role_result = subprocess.run(
    [
        AZ, "role", "assignment", "create",
        "--assignee", user_id,
        "--role", "Storage Blob Data Contributor",
        "--scope", storage_scope,
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
)
if role_result.returncode == 0:
    print("Storage Blob Data Contributor assigned to your user.")
elif "already exists" in role_result.stderr.lower():
    print("Storage Blob Data Contributor already assigned to your user.")
else:
    print("Role assignment warning:", role_result.stderr.strip())

print("RBAC can take 1-2 minutes to propagate. If the next operation returns 403, wait and retry this cell.")
time.sleep(15)

blob_service_client = BlobServiceClient(
    account_url=f"https://{storage_account_name}.blob.core.windows.net",
    credential=credential,
)

container_client = blob_service_client.get_container_client(container_name)
if container_client.exists():
    print(f"Blob container already exists: {container_name}")
else:
    container_client.create_container()
    print(f"Blob container created: {container_name}")


### Step 5: Upload Documents to Blob Storage

We upload the two sample documents from the `sample-docs/` folder into the blob container. These are the files the knowledge base will index and make searchable.

In [ ]:
# Upload sample documents to the blob container
from pathlib import Path

container_client = blob_service_client.get_container_client(container_name)

sample_dir = Path("sample-docs")
if not sample_dir.exists():
    repo_relative = Path("Foundry-IQ/Knowledge-Base/sample-docs")
    if repo_relative.exists():
        sample_dir = repo_relative
    else:
        raise FileNotFoundError(
            "sample-docs folder not found. Run the notebook from the Knowledge-Base folder "
            "or from the repository root."
        )

sample_files = ["product-faq.txt", "support-guide.txt"]

for filename in sample_files:
    filepath = sample_dir / filename
    with filepath.open("rb") as f:
        container_client.upload_blob(name=filename, data=f, overwrite=True)
    print(f"Uploaded: {filename}")

print(f"\nAll {len(sample_files)} documents uploaded to '{container_name}' container.")


### Step 6: Create an Azure AI Search Service

The Search service provides the indexing and retrieval engine behind Foundry IQ. We create a Basic-tier service with a system-assigned managed identity — this identity is what Search uses to read documents from the Storage Account.

> **Note:** Search service names must be globally unique. If creation fails with a naming conflict, change `SEARCH_SERVICE_NAME` in your `.env` file.

In [ ]:
from azure.mgmt.search import SearchManagementClient
from azure.mgmt.search.models import (
    SearchService,
    Sku as SearchSku,
    IdentityType,
    Identity,
    DataPlaneAuthOptions,
    DataPlaneAadOrApiKeyAuthOption,
)

# Read the Azure region from the environment (defaults to eastus2 if not set)
location = os.getenv("AZURE_LOCATION", "eastus2")

# Create a management client for Search operations
search_mgmt_client = SearchManagementClient(credential, subscription_id)

# Create the Search service with a system-assigned managed identity.
# auth_options enables Microsoft Entra ID (RBAC) for data-plane access — this
# notebook authenticates every index/indexer/query call with DefaultAzureCredential,
# so the service must accept Entra ID tokens (not API keys only).
print(f"Creating search service '{search_service_name}'...")
poller = search_mgmt_client.services.begin_create_or_update(
    resource_group_name=resource_group,
    search_service_name=search_service_name,
    service=SearchService(
        location=location,
        sku=SearchSku(name="basic"),
        replica_count=1,
        partition_count=1,
        identity=Identity(type=IdentityType.SYSTEM_ASSIGNED),
        auth_options=DataPlaneAuthOptions(
            aad_or_api_key=DataPlaneAadOrApiKeyAuthOption(
                aad_auth_failure_mode="http401WithBearerChallenge"
            )
        ),
    ),
)
search_service = poller.result()
print(f"Search service created: {search_service.name}")
print(f"  Endpoint: https://{search_service.name}.search.windows.net")
print(f"  Managed identity: {search_service.identity.principal_id}")

### Step 7: Assign RBAC Roles

Three role assignments are required for identity-based access:

1. **Storage Blob Data Reader** — granted to the Search service's managed identity so it can read documents from the blob container during indexing.
2. **Search Index Data Contributor** — granted to your user account so you can create indexes, data sources, and indexers via the SDK.
3. **Search Service Contributor** — also granted to your user account; required to create and manage the knowledge base / index resources in later steps.

We use the Azure CLI for role assignments because it is simpler than the authorization management SDK for this task.

> **Note:** RBAC role assignments can take 1 to 2 minutes to propagate. If subsequent steps fail with a 403 error, wait a moment and retry.

In [ ]:
import subprocess
import shutil

# Resolve the Azure CLI binary path
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError(
        "Azure CLI (az) not found. "
        "Install it from https://learn.microsoft.com/cli/azure/install-azure-cli"
    )

# Get the Search service managed identity principal ID
search_mi = search_service.identity.principal_id

# Get your signed-in user ID
user_result = subprocess.run(
    [AZ, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8",
)
user_id = user_result.stdout.strip()
print(f"Your user ID: {user_id}")
print(f"Search MI:     {search_mi}")

# Build the scope for the storage account
storage_scope = (
    f"/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}"
    f"/providers/Microsoft.Storage/storageAccounts/{storage_account_name}"
)

# Build the scope for the search service
search_scope = (
    f"/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}"
    f"/providers/Microsoft.Search/searchServices/{search_service_name}"
)

# Role assignments: (role name, assignee, principal type, scope)
roles = [
    # Search MI needs to read blobs from the storage account
    ("Storage Blob Data Reader", search_mi, "ServicePrincipal", storage_scope),
    # Your user needs to manage indexes, data sources, indexers, knowledge sources, and knowledge bases
    ("Search Index Data Contributor", user_id, "User", search_scope),
    ("Search Service Contributor", user_id, "User", search_scope),
]

for role_name, assignee, principal_type, scope in roles:
    cmd = [
        AZ, "role", "assignment", "create",
        "--assignee-object-id", assignee,
        "--assignee-principal-type", principal_type,
        "--role", role_name,
        "--scope", scope,
    ]
    r = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8")
    if r.returncode == 0:
        print(f"  [OK] {role_name} -> {principal_type}")
    elif "already exists" in r.stderr.lower():
        print(f"  [OK] {role_name} already assigned -> {principal_type}")
    else:
        print(f"  [FAILED] {role_name} -> {principal_type}")
        print(f"        {r.stderr.strip()}")

### Step 8: Create the Search Indexing Pipeline

The search indexing pipeline consists of three components:

1. **Index** — defines the schema (which fields exist, which are searchable)
2. **Data source connection** — tells Search where to find the documents (our blob container)
3. **Indexer** — the pipeline that reads documents from the data source, extracts content, and writes it into the index

We create all three using the Azure AI Search data-plane SDK. The data source connection uses a managed identity resource ID (not a connection string with keys) so that Search authenticates to Storage via RBAC.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (
    SearchIndex,
    SearchField,
    SearchFieldDataType,
    SearchIndexerDataSourceConnection,
    SearchIndexerDataContainer,
    SearchIndexer,
    FieldMapping,
    FieldMappingFunction,
    SemanticSearch,
    SemanticConfiguration,
    SemanticPrioritizedFields,
    SemanticField,
)

search_endpoint = f"https://{search_service_name}.search.windows.net"

# --- 1. Create the search index ---
index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)

index = SearchIndex(
    name=index_name,
    fields=[
        # Unique document key (base64-encoded storage path)
        SearchField(
            name="id",
            type=SearchFieldDataType.String,
            key=True,
            filterable=True,
        ),
        # Full text content extracted from the document
        SearchField(
            name="content",
            type=SearchFieldDataType.String,
            searchable=True,
        ),
        # Original storage path for reference
        SearchField(
            name="metadata_storage_path",
            type=SearchFieldDataType.String,
            filterable=True,
        ),
        # Document file name
        SearchField(
            name="metadata_storage_name",
            type=SearchFieldDataType.String,
            searchable=True,
            filterable=True,
        ),
    ],
    # A Foundry IQ knowledge source requires the target index to have a
    # semantic configuration — agentic retrieval relies on the semantic ranker.
    semantic_search=SemanticSearch(
        configurations=[
            SemanticConfiguration(
                name="default",
                prioritized_fields=SemanticPrioritizedFields(
                    title_field=SemanticField(field_name="metadata_storage_name"),
                    content_fields=[SemanticField(field_name="content")],
                ),
            )
        ]
    ),
)

index_client.create_or_update_index(index)
print(f"Index ready: {index_name}")

In [ ]:
# --- 2. Create the data source connection ---
indexer_client = SearchIndexerClient(endpoint=search_endpoint, credential=credential)

# Use a managed identity resource ID instead of a storage key
# This tells Search to authenticate to Storage using its system-assigned MI
connection_string = (
    f"ResourceId=/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group}"
    f"/providers/Microsoft.Storage/storageAccounts/{storage_account_name};"
)

data_source = SearchIndexerDataSourceConnection(
    name=data_source_name,
    type="azureblob",
    connection_string=connection_string,
    container=SearchIndexerDataContainer(name=container_name),
)

indexer_client.create_or_update_data_source_connection(data_source)
print(f"Data source ready: {data_source.name}")

In [ ]:
# --- 3. Create the indexer ---
# The indexer connects the data source to the index and runs the extraction pipeline.
# Field mappings tell the indexer how to map blob metadata to index fields.
indexer = SearchIndexer(
    name=indexer_name,
    data_source_name=data_source_name,
    target_index_name=index_name,
    field_mappings=[
        # Map the base64-encoded storage path to the document key
        FieldMapping(
            source_field_name="metadata_storage_path",
            target_field_name="id",
            mapping_function=FieldMappingFunction(name="base64Encode"),
        ),
        # Map the file name for filtering and display
        FieldMapping(
            source_field_name="metadata_storage_name",
            target_field_name="metadata_storage_name",
        ),
    ],
)

indexer_client.create_or_update_indexer(indexer)
print(f"Indexer ready: {indexer.name}")
print("\nKnowledge base infrastructure is ready.")

### Step 9: Run the Indexer and Wait for Completion

Creating the indexer does not automatically start it. We trigger a run and then poll the indexer status until it finishes processing the documents.

In [ ]:
import time
from azure.core.exceptions import HttpResponseError

# Trigger the indexer to start processing documents.
try:
    indexer_client.run_indexer(indexer_name)
    print("Indexer started. Waiting for completion...")
except HttpResponseError as ex:
    if "already running" in str(ex).lower():
        print("Indexer is already running. Waiting for the current run to finish...")
    else:
        raise

# Poll until the indexer finishes
while True:
    status = indexer_client.get_indexer_status(indexer_name)
    last_result = status.last_result

    if last_result and last_result.status in (
        "success",
        "transientFailure",
        "persistentFailure",
    ):
        print(f"\nIndexer finished.")
        print(f"  Status:              {last_result.status}")
        print(f"  Documents processed: {last_result.item_count}")
        print(f"  Documents failed:    {last_result.failed_item_count}")
        if last_result.errors:
            for error in last_result.errors:
                print(f"  Error: {error.error_message}")
        if last_result.status == "persistentFailure" or (last_result.failed_item_count or 0) > 0:
            raise RuntimeError("Indexer completed with failures. Review the errors above before continuing.")
        break

    print("  Indexing in progress...")
    time.sleep(10)


### Step 10: Create the Azure AI Search Knowledge Source and Knowledge Base

The index is now populated. Next we create an Azure AI Search **knowledge source** that points at the index, then a **knowledge base** that exposes the MCP endpoint used by Foundry IQ agents.


In [ ]:
import requests

api_version = "2025-11-01-preview"
search_token = credential.get_token("https://search.azure.com/.default").token
headers = {
    "Authorization": f"Bearer {search_token}",
    "Content-Type": "application/json",
    "Accept": "application/json;odata.metadata=minimal",
    "Prefer": "return=representation",
}

knowledge_source_payload = {
    "name": knowledge_source_name,
    "kind": "searchIndex",
    "description": "Knowledge source backed by the support document search index.",
    "searchIndexParameters": {
        "searchIndexName": index_name,
        "searchFields": [{"name": "content"}],
        "sourceDataFields": [
            {"name": "metadata_storage_name"},
            {"name": "metadata_storage_path"},
        ],
    },
}

ks_url = f"{search_endpoint}/knowledgesources('{knowledge_source_name}')?api-version={api_version}"
ks_response = requests.put(ks_url, headers=headers, json=knowledge_source_payload)
ks_response.raise_for_status()
print(f"Knowledge source ready: {knowledge_source_name}")

knowledge_base_payload = {
    "name": knowledge_base_name,
    "description": "Knowledge base over the uploaded support documents.",
    "knowledgeSources": [{"name": knowledge_source_name}],
    "outputMode": "extractiveData",
    "retrievalReasoningEffort": {"kind": "minimal"},
}

kb_url = f"{search_endpoint}/knowledgebases('{knowledge_base_name}')?api-version={api_version}"
kb_response = requests.put(kb_url, headers=headers, json=knowledge_base_payload)
kb_response.raise_for_status()

kb_mcp_url = f"{search_endpoint}/knowledgebases/{knowledge_base_name}/mcp?api-version={api_version}"
print(f"Knowledge base ready: {knowledge_base_name}")
print(f"MCP URL: {kb_mcp_url}")


### Step 11: Register the Knowledge Base with the Foundry Project

The knowledge base now lives on the Search service, but the Foundry project cannot see it yet. The **Knowledge (Foundry IQ)** blade in the portal — and any agent you create in the portal — list knowledge bases through an **Azure AI Search connection** on the project. Without this connection, the knowledge base you just built will not appear in the portal even though it exists.

The agent you build in code reaches the knowledge base directly over its MCP endpoint, so it does not need this connection. The portal experience does. We register the connection with **Microsoft Entra ID** authentication — no admin keys — and the portal enumerates the knowledge base using your signed-in identity, which already holds the Search roles granted in Step 7.

In [ ]:
# Register the Search service as an Azure AI Search connection on the Foundry project.
# The knowledge base lives on the Search service, but the portal's Knowledge (Foundry IQ)
# blade — and any agent you build in the portal — list knowledge bases through an
# Azure AI Search connection on the project. The code-first agent reaches the knowledge
# base over its MCP endpoint and does not need this; the portal experience does.
import requests
from azure.identity import get_bearer_token_provider

# Name of the connection as it appears in the project (override in .env if you like)
search_connection_name = os.getenv("SEARCH_PROJECT_CONNECTION_NAME", "support-kb-search")

# The Foundry project ARM resource ID, derived from the project endpoint:
#   https://<account>.services.ai.azure.com/api/projects/<project>
# Override with FOUNDRY_PROJECT_RESOURCE_ID in .env if your Foundry account lives in a
# different resource group than the one used for storage and search above.
project_resource_id = os.getenv("FOUNDRY_PROJECT_RESOURCE_ID")
if not project_resource_id:
    account_name = endpoint.split("://", 1)[1].split(".", 1)[0]
    project_name = endpoint.rstrip("/").rsplit("/", 1)[-1]
    project_resource_id = (
        f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
        f"/providers/Microsoft.CognitiveServices/accounts/{account_name}/projects/{project_name}"
    )

search_resource_id = (
    f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
    f"/providers/Microsoft.Search/searchServices/{search_service_name}"
)

# Management-plane token from the same credential used throughout the notebook
mgmt_token = get_bearer_token_provider(credential, "https://management.azure.com/.default")

# authType AAD = Microsoft Entra ID: no admin keys are stored on the connection. The
# portal enumerates knowledge bases with your signed-in identity, which already holds
# the Search Index Data Contributor and Search Service Contributor roles from Step 7.
connection_payload = {
    "name": search_connection_name,
    "properties": {
        "authType": "AAD",
        "category": "CognitiveSearch",
        "isSharedToAll": True,
        "target": search_endpoint,
        "metadata": {
            "ApiType": "Azure",
            "ApiVersion": "2024-05-01-preview",
            "DeploymentApiVersion": "2023-11-01",
            "ResourceId": search_resource_id,
            "type": "azure_ai_search",
        },
    },
}

connection_url = (
    f"https://management.azure.com{project_resource_id}"
    f"/connections/{search_connection_name}?api-version=2025-10-01-preview"
)
connection_response = requests.put(
    connection_url,
    headers={
        "Authorization": f"Bearer {mgmt_token()}",
        "Content-Type": "application/json",
    },
    json=connection_payload,
)
connection_response.raise_for_status()
print(f"Azure AI Search connection registered on the project: {search_connection_name}")
print("In the portal, open Knowledge (Foundry IQ), pick this connection, and the knowledge base appears.")

### Step 12: Verify the Knowledge Base

We run a test search query against the index to confirm that documents were indexed correctly and are returning relevant results. If this step returns results with content from your uploaded documents, the knowledge base is ready for use with Foundry IQ agents.

In [ ]:
from azure.search.documents import SearchClient

# Create a data-plane search client to query the index
search_data_client = SearchClient(
    endpoint=search_endpoint,
    index_name=index_name,
    credential=credential,
)

# Run a test query against the knowledge base
print("Search query: 'warranty policy'\n")
results = search_data_client.search(search_text="warranty policy", top=3)

found = False
for result in results:
    found = True
    score = result["@search.score"]
    content = result.get("content", "")
    filename = result.get("metadata_storage_name", "unknown")
    # Show the first 300 characters of the matched content
    preview = content[:300].replace("\n", " ")
    print(f"File:    {filename}")
    print(f"Score:   {score:.2f}")
    print(f"Content: {preview}...")
    print()

if not found:
    print("No results found. The indexer may still be running — wait a moment and retry.")

In [ ]:
# Run a second query to test a different topic
print("Search query: 'troubleshooting speaker'\n")
results = search_data_client.search(search_text="troubleshooting speaker", top=3)

for result in results:
    score = result["@search.score"]
    content = result.get("content", "")
    filename = result.get("metadata_storage_name", "unknown")
    preview = content[:300].replace("\n", " ")
    print(f"File:    {filename}")
    print(f"Score:   {score:.2f}")
    print(f"Content: {preview}...")
    print()

---

### Summary

In this notebook you:

1. **Created a Storage Account** and a blob container to hold your knowledge source documents
2. **Uploaded documents** — two sample text files — to the container
3. **Created an Azure AI Search service** with a system-assigned managed identity
4. **Assigned RBAC roles** so Search can read from Storage and your user can manage the index
5. **Built the search infrastructure** — an index, a data source connection, and an indexer
6. **Ran the indexer** so the documents are indexed and searchable
7. **Created the knowledge source and knowledge base** and printed the MCP endpoint Foundry IQ agents use
8. **Registered the knowledge base with the Foundry project** as an Azure AI Search connection so it appears under Knowledge (Foundry IQ) in the portal
9. **Verified retrieval** with test search queries against the index

This knowledge base is now ready to be used as a grounding source for Foundry IQ agents. An agent connects to the knowledge base MCP endpoint shown above and answers questions grounded in your documents. Because you registered the Azure AI Search connection, the knowledge base also appears in the portal's Knowledge (Foundry IQ) blade, where you can attach it to an agent without writing code.